In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
# magic autoreload
%load_ext autoreload
%autoreload 2
from svd_imputer import Imputer
from sklearn.preprocessing import StandardScaler

In [ ]:
fname = os.path.join("data","serie_17092025225957.csv")
df = pd.read_csv(fname,index_col=0)
df.index.name = "date"
df.drop(columns=[i for i in df.columns if "Unnamed:" in i],inplace=True)
# drop if column ha sless than x values
df = df.drop(columns='593/5')
df = df.loc[df['594/400']>0]
df = df.loc[df['602/187']<5]
df = df.dropna(thresh=30,axis=1)
df.info()


In [ ]:
df.index = pd.to_datetime(df.index, format="%d/%m/%Y %H:%M")
df

In [ ]:
# get moving average
#df = df.rolling(window=3, min_periods=3).mean()

In [ ]:
df = df.resample('ME').mean()
for col in df.columns:
    fig,ax = plt.subplots()
    ax.plot(df.index,df[col],'o-')
    ax.set_title(col)
    plt.show()

In [ ]:
df.shape

In [ ]:
df = df.dropna(how='all',axis=0)

In [ ]:
data = df.copy()

In [ ]:


def create_derivative_augmented_matrix(X):
    """
    Augment with first and second differences.
    
    [X_t, ΔX_t, Δ²X_t]
    """
    dX = np.diff(X, axis=0)
    ddX = np.diff(dX, axis=0)
    
    # Align dimensions
    n_valid = len(ddX)
    X_aligned = X[2:2+n_valid, :]
    dX_aligned = dX[1:1+n_valid, :]
    
    X_aug = np.hstack([X_aligned, dX_aligned, ddX])
    return X_aug


def create_symmetric_augmented_matrix(X, window=3):
    """
    Include past and future lags: [X_{t-w}, ..., X_{t}, ..., X_{t+w}]
    
    Better for interpolation (gaps in middle of series).
    """
    lags = list(range(-window, window + 1))
    # lags = [-3, -2, -1, 0, 1, 2, 3]
    
    n_samples, n_features = X.shape
    n_valid = n_samples - 2 * window
    
    X_aug = np.full((n_valid, n_features * len(lags)), np.nan)
    
    for i, lag in enumerate(lags):
        start_row = window + lag
        end_row = start_row + n_valid
        col_start = i * n_features
        col_end = (i + 1) * n_features
        
        X_aug[:, col_start:col_end] = X[start_row:end_row, :]
    
    return X_aug


data_expanded = data.copy()
data_expanded = create_derivative_augmented_matrix(data_expanded.values)
data_expanded = pd.DataFrame(data_expanded,
                             index=data.index[2:2+data_expanded.shape[0]],
                             columns=[f"{col}_orig" for col in data.columns] +
                                     [f"{col}_d1" for col in data.columns] +
                                     [f"{col}_d2" for col in data.columns])
#data_expanded = create_symmetric_augmented_matrix(data.values, window=3)
#data_expanded = pd.DataFrame(data_expanded,
#                             index=data.index[3:3+data_expanded.shape[0]],
#                             columns=[f"{col}_lag{lag}" for lag in range(-3,4) for col in data.columns])
data_expanded.shape,data.shape


In [ ]:
imputer = Imputer(data_expanded,
                  variance_threshold=.999,tol=1e-3,verbose=True)      # validate_dataframe() + preprocessing ONCE
imputer.fit()                        # Pure computation, cached SVD components
results = imputer.transform()        # Uses cached data + SVD components  
unc = imputer.estimate_uncertainty(frac=0.01)  # Uses cached data
#new_projected = imputer.project_data(data_expanded)    # NEW: SVD projection
new_reconstructed = imputer.reconstruct_data()  # NEW: SVD reconstruction

In [ ]:
residuals = new_reconstructed.iloc[:,:data.shape[1]] - data_expanded.iloc[:,:data.shape[1]]
residuals = imputer.calculate_reconstruction_residuals(return_stats=False)
residuals.iloc[:,:data.shape[1]].plot()

In [ ]:


#imputer = Imputer(variance_threshold=.95,tol=1e-3, verbose=True)
#df_imputed, unc = imputer.fit_transform(data_expanded,
#                                        return_uncertainty=True,
#                                        #uncertainty_method='monte_carlo',
#                                        #n_bootstrap=100,
#                                        n_repeats=50,
#                                        mask_strategy='random',
#                                        frac=.1)
#df_lower, df_upper = imputer.get_confidence_intervals(df_imputed, unc)




In [ ]:
data_ = data_expanded.dropna(how="all")

mcres = np.array(unc['raw_imputed'])
mcres[:,:,:].shape

In [ ]:

fig,axs = plt.subplots(df.shape[1],1, figsize=(7,df.shape[1]*3),sharex=True)

for e,ax in enumerate(axs):
    ax.plot(data_.index,data_.iloc[:,e],'r.',zorder=1,lw=1)
    ax.axhline(data_.iloc[:,e].mean(),c='k',linestyle='--')
    ax.plot(results.index,results.iloc[:,e],'b-.',zorder=1,lw=1)
    [ax.plot(data_.index, mcres[i,:,e],c="0.5",alpha=0.3,zorder=0) for i in range(mcres.shape[0])]
fig.tight_layout()